# Ejercicio Práctico: Despliegue completo de un modelo de Machine Learning

> **Curso:** Aplicaciones de Inteligencia Artificial II  
> **Autor:** Marlon Cárdenas Bonett  
> **Duración:** 2 horas  
> **Nivel:** Universitario / Máster  

## 🎯 Objetivo del ejercicio

En este ejercicio práctico, implementarás **desde cero** un sistema completo de Machine Learning en producción:

1. ✅ Entrenar un modelo de clasificación (predicción de enfermedad cardíaca)
2. ✅ Crear un wrapper para encapsular la lógica de inferencia
3. ✅ Desarrollar una API REST con FastAPI
4. ✅ Contenerizar la aplicación con Docker
5. ✅ Implementar monitoreo básico con Prometheus

Al finalizar, tendrás un **sistema completo listo para producción** con todas las mejores prácticas de MLOps.

## ⏱️ Distribución del tiempo

| Parte | Tema | Tiempo |
|-------|------|--------|
| **1** | Entrenamiento y Empaquetado | 30 min |
| **2** | API con FastAPI | 30 min |
| **3** | Dockerización | 40 min |
| **4** | Monitoreo Básico | 20 min |

## 📋 Pre-requisitos

Antes de comenzar, verifica que tienes instalado:

**Software necesario:**
- Python 3.8 o superior
- Docker Desktop (para Windows/Mac) o Docker Engine (Linux)
- Jupyter Notebook o VS Code con extensión de Python

**Librerías Python:**
- scikit-learn
- pandas
- numpy
- fastapi
- uvicorn
- joblib
- prometheus-client
- requests

## ✅ Verificación de Pre-requisitos

Ejecuta las siguientes celdas para verificar que todo está listo:

In [14]:
# Verificación de versión de Python
import sys
print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")

assert sys.version_info >= (3, 8), "❌ Necesitas Python 3.8 o superior"
print("✅ Python version OK")

Python version: 3.12.9 (tags/v3.12.9:fdb8142, Feb  4 2025, 15:27:58) [MSC v.1942 64 bit (AMD64)]
Python executable: c:\cosas uni\tercero\ia\venv\Scripts\python.exe
✅ Python version OK


In [15]:
# Verificación de librerías
import importlib

libraries = [
    'sklearn',
    'pandas',
    'numpy',
    'fastapi',
    'uvicorn',
    'joblib',
    'prometheus_client',
    'requests'
]

for lib in libraries:
    try:
        module = importlib.import_module(lib)
        version = getattr(module, '__version__', 'unknown')
        print(f"✅ {lib}: {version}")
    except ImportError:
        print(f"❌ {lib}: NO INSTALADO")

✅ sklearn: 1.8.0
✅ pandas: 3.0.2
✅ numpy: 2.4.4
❌ fastapi: NO INSTALADO
❌ uvicorn: NO INSTALADO
✅ joblib: 1.5.3
✅ prometheus_client: unknown
✅ requests: 2.33.1


In [16]:
# Verificación de Docker (ejecuta esto en terminal, no aquí)
# Copia y pega en tu terminal:
# docker --version
# docker ps

print("⚠️ IMPORTANTE: Abre una terminal y ejecuta:")
print("   docker --version")
print("   docker ps")
print("")
print("Si ves la versión de Docker y no hay errores, estás listo.")

⚠️ IMPORTANTE: Abre una terminal y ejecuta:
   docker --version
   docker ps

Si ves la versión de Docker y no hay errores, estás listo.


## 📁 Estructura del Proyecto

Durante este ejercicio, crearemos la siguiente estructura de archivos:

```
Tema_07/
├── Ejercicio_Practico_Despliegue_ML.ipynb  ← Estás aquí
├── heart_model_wrapper.py                  ← Wrapper del modelo
├── api_heart.py                            ← API FastAPI
├── heart_disease_model.joblib              ← Modelo serializado
├── requirements.txt                         ← Dependencias
├── Dockerfile                               ← Imagen Docker
├── .dockerignore                            ← Archivos a ignorar
├── docker-compose.yml                       ← Orquestación (opcional)
└── prometheus.yml                           ← Config Prometheus (opcional)
```

# Parte 1: Entrenamiento y Empaquetado del Modelo (30 min)

## Contexto del Problema

Vamos a construir un **sistema de predicción de enfermedad cardíaca** basado en características clínicas de pacientes. Este es un problema de **clasificación binaria**:

- **Clase 0:** Paciente sin enfermedad cardíaca
- **Clase 1:** Paciente con enfermedad cardíaca

El dataset contiene 13 características clínicas como edad, sexo, presión arterial, colesterol, etc.

## 📝 Paso 1.1: Cargar y Explorar los Datos

**Tarea:** Cargar el dataset Heart Disease de UCI y hacer una exploración inicial.

**Instrucciones:**
1. Ejecuta la siguiente celda para cargar los datos
2. Observa la estructura del dataset
3. Verifica que no hay valores nulos

In [17]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
import warnings
warnings.filterwarnings('ignore')

# Cargar dataset Heart Disease de OpenML
print("📥 Descargando dataset Heart Disease...")

# Opción 1: Usar OpenML (requiere internet)
try:
    data = fetch_openml('heart-statlog', version=1, as_frame=True, parser='auto')
    df = data.frame
    print("✅ Dataset cargado desde OpenML")
except Exception as e:
    print(f"⚠️ Error descargando de OpenML: {e}")
    print("📦 Creando dataset sintético para el ejercicio...")
    
    # Dataset sintético si no hay internet
    np.random.seed(42)
    n_samples = 270
    
    df = pd.DataFrame({
        'age': np.random.randint(29, 78, n_samples),
        'sex': np.random.choice([0, 1], n_samples),
        'chest_pain': np.random.choice([1, 2, 3, 4], n_samples),
        'rest_blood_pressure': np.random.randint(94, 200, n_samples),
        'cholesterol': np.random.randint(126, 565, n_samples),
        'fasting_blood_sugar': np.random.choice([0, 1], n_samples),
        'rest_ecg': np.random.choice([0, 1, 2], n_samples),
        'max_heart_rate': np.random.randint(71, 202, n_samples),
        'exercise_angina': np.random.choice([0, 1], n_samples),
        'oldpeak': np.random.uniform(0, 6.2, n_samples),
        'slope': np.random.choice([1, 2, 3], n_samples),
        'vessels': np.random.choice([0, 1, 2, 3], n_samples),
        'thal': np.random.choice([3, 6, 7], n_samples),
        'class': np.random.choice([0, 1], n_samples)
    })
    print("✅ Dataset sintético creado")

# Renombrar columna target si es necesario
if 'class' not in df.columns and 'target' in df.columns:
    df.rename(columns={'target': 'class'}, inplace=True)

print(f"\n📊 Dimensiones del dataset: {df.shape}")
print(f"   - {df.shape[0]} pacientes")
print(f"   - {df.shape[1]-1} características + 1 target\n")

df.head()

📥 Descargando dataset Heart Disease...
✅ Dataset cargado desde OpenML

📊 Dimensiones del dataset: (270, 14)
   - 270 pacientes
   - 13 características + 1 target



,age,sex,chest,resting_blood_pressure,serum_cholestoral,fasting_blood_sugar,resting_electrocardiographic_results,maximum_heart_rate_achieved,exercise_induced_angina,oldpeak,slope,number_of_major_vessels,thal,class
0,70,1,4,130,322,0,2,109,0,2.4,2,3,3,present
1,67,0,3,115,564,0,2,160,0,1.6,2,0,7,absent
2,57,1,2,124,261,0,0,141,0,0.3,1,0,7,present
3,64,1,4,128,263,0,0,105,1,0.2,2,1,7,absent
4,74,0,2,120,269,0,2,121,1,0.2,1,1,3,absent


In [18]:
# Exploración rápida de datos
print("📋 Información del Dataset:")
print(df.info())
print("\n" + "="*60)

print("\n🔍 Estadísticas Descriptivas:")
print(df.describe())
print("\n" + "="*60)

print("\n❓ Valores Nulos:")
print(df.isnull().sum())
print("\n" + "="*60)

print("\n🎯 Distribución de la Variable Target:")
print(df['class'].value_counts())
print(f"\nProporción:")
print(df['class'].value_counts(normalize=True))

📋 Información del Dataset:
<class 'pandas.DataFrame'>
RangeIndex: 270 entries, 0 to 269
Data columns (total 14 columns):
 #   Column                                Non-Null Count  Dtype   
---  ------                                --------------  -----   
 0   age                                   270 non-null    int64   
 1   sex                                   270 non-null    int64   
 2   chest                                 270 non-null    int64   
 3   resting_blood_pressure                270 non-null    int64   
 4   serum_cholestoral                     270 non-null    int64   
 5   fasting_blood_sugar                   270 non-null    int64   
 6   resting_electrocardiographic_results  270 non-null    int64   
 7   maximum_heart_rate_achieved           270 non-null    int64   
 8   exercise_induced_angina               270 non-null    int64   
 9   oldpeak                               270 non-null    float64 
 10  slope                                 270 non-null    int6

### ✅ Checkpoint 1.1

**Deberías ver:**
- ✓ Dataset con ~270 filas y 13-14 columnas
- ✓ Sin valores nulos
- ✓ Columna `class` con valores 0 y 1
- ✓ Distribución balanceada entre clases (aprox. 50-50)

**Si hay problemas:**
- Si falla la descarga, el código creará un dataset sintético automáticamente
- Si hay valores nulos, puedes eliminarlos con: `df.dropna(inplace=True)`

## 📝 Paso 1.2: Preparar los Datos para Entrenamiento

**Tarea:** Separar características (X) y target (y), y dividir en conjuntos de entrenamiento y prueba.

**Instrucciones:**
1. Ejecuta la celda para separar X e y
2. Divide en train/test con 80/20
3. Verifica los tamaños de los conjuntos

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Separar características y target
X = df.drop('class', axis=1)
y = df['class']

# Convertir características a numérico si es necesario
X = X.apply(pd.to_numeric, errors='coerce')

# Codificar la variable target si contiene valores categóricos
if y.dtype == 'object' or isinstance(y.dtype, pd.CategoricalDtype):
    print("🔄 Codificando variable target categórica...")
    le = LabelEncoder()
    y = le.fit_transform(y)
    print(f"   Mapeo: {dict(zip(le.classes_, le.transform(le.classes_)))}")
    y = pd.Series(y, index=df.index)  # Mantener el índice
else:
    y = y.astype(int)

print("\n📊 Características (X):")
print(f"   Shape: {X.shape}")
print(f"   Columnas: {list(X.columns)}")

print("\n🎯 Target (y):")
print(f"   Shape: {y.shape}")
print(f"   Valores únicos: {sorted(y.unique())}")
print(f"   Tipo: {y.dtype}")

# División train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  # Mantener proporción de clases
)

print("\n✂️ División Train/Test:")
print(f"   Train: {X_train.shape[0]} muestras")
print(f"   Test:  {X_test.shape[0]} muestras")
print(f"   Proporción: {X_train.shape[0]/len(X)*100:.1f}% / {X_test.shape[0]/len(X)*100:.1f}%")

🔄 Codificando variable target categórica...
   Mapeo: {'absent': np.int64(0), 'present': np.int64(1)}

📊 Características (X):
   Shape: (270, 13)
   Columnas: ['age', 'sex', 'chest', 'resting_blood_pressure', 'serum_cholestoral', 'fasting_blood_sugar', 'resting_electrocardiographic_results', 'maximum_heart_rate_achieved', 'exercise_induced_angina', 'oldpeak', 'slope', 'number_of_major_vessels', 'thal']

🎯 Target (y):
   Shape: (270,)
   Valores únicos: [np.int64(0), np.int64(1)]
   Tipo: int64

✂️ División Train/Test:
   Train: 216 muestras
   Test:  54 muestras
   Proporción: 80.0% / 20.0%


### ✅ Checkpoint 1.2

**Deberías ver:**
- ✓ X_train con ~216 muestras (80%)
- ✓ X_test con ~54 muestras (20%)
- ✓ Todas las columnas son numéricas

## 📝 Paso 1.3: Crear y Entrenar el Pipeline

**Tarea:** Crear un pipeline con preprocesamiento y modelo, luego entrenarlo.

**Instrucciones:**
1. Crea un Pipeline con StandardScaler + LogisticRegression
2. Entrena el modelo
3. Observa el tiempo de entrenamiento

*¿Qué es exactamente L‑BFGS?*
- L‑BFGS viene de Limited-memory Broyden–Fletcher–Goldfarb–Shanno.

- Es un método de optimización quasi‑Newton, que aproxima la inversa de la Hessiana (segunda derivada) sin almacenarla entera, lo que lo hace eficiente en problemas con muchos parámetros.

- Se usa para minimizar una función diferenciable (en este caso, la función de coste de la regresión logística con regularización L2)

In [20]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import time

print("🏗️ Construyendo el Pipeline...\n")

# Crear el pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),           # Paso 1: Normalización
    ('classifier', LogisticRegression(      # Paso 2: Clasificador
        max_iter=1000,
        random_state=42,
        solver='lbfgs'
    ))
])

print("Pipeline creado:")
print(pipeline)
print("\n" + "="*60)

# Entrenar el modelo
print("\n🚀 Entrenando el modelo...")
start_time = time.time()

pipeline.fit(X_train, y_train)

training_time = time.time() - start_time

print(f"✅ Modelo entrenado en {training_time:.2f} segundos")
print("\n" + "="*60)

🏗️ Construyendo el Pipeline...

Pipeline creado:
Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(max_iter=1000, random_state=42))])


🚀 Entrenando el modelo...
✅ Modelo entrenado en 0.01 segundos



### ✅ Checkpoint 1.3

**Deberías ver:**
- ✓ Pipeline con 2 pasos: scaler y classifier
- ✓ Entrenamiento completado en < 1 segundo
- ✓ Sin errores ni warnings importantes

## 📝 Paso 1.4: Evaluar el Modelo

**Tarea:** Evaluar el rendimiento del modelo con métricas clave.

**Instrucciones:**
1. Hacer predicciones en el conjunto de test
2. Calcular accuracy, precision, recall, F1
3. Mostrar matriz de confusión

In [21]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, confusion_matrix, classification_report
)

# Predicciones
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)

# Métricas
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("📊 RESULTADOS DE EVALUACIÓN\n")
print("="*60)
print(f"  Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"  Recall:    {recall:.4f} ({recall*100:.2f}%)")
print(f"  F1-Score:  {f1:.4f}")
print("="*60)

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
print("\n📈 Matriz de Confusión:\n")
print("                Predicho")
print("              0 (No)  1 (Sí)")
print(f"Real  0 (No)   {cm[0,0]:3d}     {cm[0,1]:3d}")
print(f"      1 (Sí)   {cm[1,0]:3d}     {cm[1,1]:3d}")

# Reporte de clasificación
print("\n📋 Reporte Detallado:\n")
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

📊 RESULTADOS DE EVALUACIÓN

  Accuracy:  0.8519 (85.19%)
  Precision: 0.7857 (78.57%)
  Recall:    0.9167 (91.67%)
  F1-Score:  0.8462

📈 Matriz de Confusión:

                Predicho
              0 (No)  1 (Sí)
Real  0 (No)    24       6
      1 (Sí)     2      22

📋 Reporte Detallado:

              precision    recall  f1-score   support

  No Disease       0.92      0.80      0.86        30
     Disease       0.79      0.92      0.85        24

    accuracy                           0.85        54
   macro avg       0.85      0.86      0.85        54
weighted avg       0.86      0.85      0.85        54



### ✅ Checkpoint 1.4

**Deberías ver:**
- ✓ Accuracy > 75% (idealmente > 80%)
- ✓ Precision y Recall balanceados
- ✓ Matriz de confusión con valores en la diagonal principal

**Si el modelo tiene bajo rendimiento:**
- Es normal en datasets pequeños o sintéticos
- Para este ejercicio, lo importante es el proceso, no el rendimiento perfecto

## 📝 Paso 1.5: Importar el Wrapper del Modelo

**Tarea:** Utilizar una clase wrapper profesional desde un módulo externo.

**¿Por qué separar el Wrapper en un archivo?**
- ✅ **Reutilización**: El mismo código se puede usar en notebooks, APIs, scripts, etc.
- ✅ **Mantenimiento**: Cambios en un solo lugar afectan a todo el proyecto
- ✅ **Testing**: Facilita las pruebas unitarias del wrapper
- ✅ **Despliegue**: Necesario para APIs (FastAPI, Flask) y Docker
- ✅ **Profesionalidad**: Refleja buenas prácticas de ingeniería de software

**Estructura del archivo `heart_model_wrapper.py`:**
- **Encapsula toda la lógica de inferencia**
  - Preprocesamiento de datos de entrada
  - Validación de características
  - Predicción del modelo
  - Postprocesamiento de resultados
- **Interfaz limpia y documentada**
  - `predict()`: Predicción individual
  - `predict_batch()`: Predicciones múltiples
  - `get_feature_info()`: Información de características

**Instrucciones:**
1. Ejecuta la celda siguiente para importar el wrapper
2. El archivo `heart_model_wrapper.py` ya está creado en la carpeta del proyecto
3. Puedes editarlo externamente y reimportar si necesitas hacer cambios

In [22]:
# Importar el Wrapper desde el módulo externo
from heart_model_wrapper import HeartDiseaseWrapper

# Verificar que la importación funcionó correctamente
print("✅ Clase HeartDiseaseWrapper importada correctamente")
print(f"   Ubicación: heart_model_wrapper.py")
print(f"   Métodos disponibles:")
print(f"      - preprocess()")
print(f"      - predict()")
print(f"      - predict_batch()")
print(f"      - postprocess()")
print(f"      - get_feature_info()")

✅ Clase HeartDiseaseWrapper importada correctamente
   Ubicación: heart_model_wrapper.py
   Métodos disponibles:
      - preprocess()
      - predict()
      - predict_batch()
      - postprocess()
      - get_feature_info()


## 📝 Paso 1.6: Probar el Wrapper

**Tarea:** Instanciar el wrapper y probar con datos de ejemplo.

**Instrucciones:**
1. Crear instancia del wrapper con el modelo entrenado
2. Probar predicción individual
3. Probar predicción batch

In [23]:
# Crear instancia del wrapper
wrapper = HeartDiseaseWrapper(pipeline)

print("🎁 Wrapper instanciado correctamente")
print(f"   Características esperadas: {len(wrapper.feature_names)}")
print(f"   Nombres: {wrapper.feature_names}")

🎁 Wrapper instanciado correctamente
   Características esperadas: 13
   Nombres: ['age', 'sex', 'chest', 'resting_blood_pressure', 'serum_cholestoral', 'fasting_blood_sugar', 'resting_electrocardiographic_results', 'maximum_heart_rate_achieved', 'exercise_induced_angina', 'oldpeak', 'slope', 'number_of_major_vessels', 'thal']


In [24]:
# Prueba 1: Predicción individual
print("🧪 Prueba 1: Predicción Individual\n")

# Paciente de ejemplo (valores típicos de riesgo alto)
# Nota: Los nombres deben coincidir exactamente con las columnas del dataset
patient = {
    'age': 63,
    'sex': 1,
    'chest': 3,
    'resting_blood_pressure': 145,
    'serum_cholestoral': 233,
    'fasting_blood_sugar': 1,
    'resting_electrocardiographic_results': 0,
    'maximum_heart_rate_achieved': 150,
    'exercise_induced_angina': 0,
    'oldpeak': 2.3,
    'slope': 1,
    'number_of_major_vessels': 0,
    'thal': 6
}

result = wrapper.predict(patient)

print("Entrada:")
print(f"   Edad: {patient['age']} años")
print(f"   Sexo: {'Masculino' if patient['sex'] == 1 else 'Femenino'}")
print(f"   Presión arterial: {patient['resting_blood_pressure']} mmHg")
print(f"   Colesterol: {patient['serum_cholestoral']} mg/dl\n")

print("Resultado:")
print(f"   Predicción: {result['prediction_label']}")
print(f"   Confianza: {result['confidence']*100:.2f}%")
print(f"   Nivel de riesgo: {result['risk_level']}")
print(f"   Probabilidades:")
print(f"      - Sin enfermedad: {result['probabilities']['no_disease']*100:.2f}%")
print(f"      - Con enfermedad: {result['probabilities']['disease']*100:.2f}%")

🧪 Prueba 1: Predicción Individual

Entrada:
   Edad: 63 años
   Sexo: Masculino
   Presión arterial: 145 mmHg
   Colesterol: 233 mg/dl

Resultado:
   Predicción: No Disease
   Confianza: 78.75%
   Nivel de riesgo: Low
   Probabilidades:
      - Sin enfermedad: 78.75%
      - Con enfermedad: 21.25%


In [25]:
# Prueba 2: Predicción batch
print("🧪 Prueba 2: Predicción Batch\n")

# Varios pacientes
patients = [
    {
        'age': 45, 'sex': 0, 'chest': 2, 'resting_blood_pressure': 120,
        'serum_cholestoral': 180, 'fasting_blood_sugar': 0, 
        'resting_electrocardiographic_results': 0,
        'maximum_heart_rate_achieved': 170, 'exercise_induced_angina': 0, 
        'oldpeak': 0.5, 'slope': 2, 'number_of_major_vessels': 0, 'thal': 3
    },
    {
        'age': 67, 'sex': 1, 'chest': 4, 'resting_blood_pressure': 160,
        'serum_cholestoral': 286, 'fasting_blood_sugar': 1, 
        'resting_electrocardiographic_results': 2,
        'maximum_heart_rate_achieved': 108, 'exercise_induced_angina': 1, 
        'oldpeak': 4.2, 'slope': 1, 'number_of_major_vessels': 3, 'thal': 7
    },
    {
        'age': 52, 'sex': 1, 'chest': 1, 'resting_blood_pressure': 130,
        'serum_cholestoral': 200, 'fasting_blood_sugar': 0, 
        'resting_electrocardiographic_results': 1,
        'maximum_heart_rate_achieved': 155, 'exercise_induced_angina': 0, 
        'oldpeak': 1.0, 'slope': 2, 'number_of_major_vessels': 1, 'thal': 6
    }
]

results = wrapper.predict_batch(patients)

print(f"Procesados {len(results)} pacientes:\n")

for i, result in enumerate(results, 1):
    print(f"Paciente {i}:")
    print(f"  Predicción: {result['prediction_label']}")
    print(f"  Riesgo: {result['risk_level']}")
    print(f"  Confianza: {result['confidence']*100:.2f}%")
    print()

🧪 Prueba 2: Predicción Batch

Procesados 3 pacientes:

Paciente 1:
  Predicción: No Disease
  Riesgo: Low
  Confianza: 99.26%

Paciente 2:
  Predicción: Disease
  Riesgo: High
  Confianza: 99.86%

Paciente 3:
  Predicción: No Disease
  Riesgo: Medium
  Confianza: 66.86%



### ✅ Checkpoint 1.6

**Deberías ver:**
- ✓ Predicción individual con resultado detallado
- ✓ Predicción batch con 3 pacientes procesados
- ✓ Cada resultado incluye: prediction_label, confidence, risk_level, probabilities
- ✓ Sin errores de ejecución

## 📝 Paso 1.7: Guardar el Modelo Serializado

**Tarea:** Serializar el pipeline completo con Joblib.

**¿Por qué Joblib?**
- Más eficiente que Pickle para objetos numpy
- Compresión incorporada
- Formato estándar para scikit-learn

**Instrucciones:**
1. Guardar el pipeline con joblib.dump()
2. Verificar que el archivo fue creado
3. Comprobar el tamaño del archivo

In [26]:
import joblib
import os

# Nombre del archivo
model_filename = 'heart_disease_model.joblib'

print("💾 Guardando modelo...")

# Guardar el pipeline completo
joblib.dump(pipeline, model_filename)

# Verificar que se guardó
if os.path.exists(model_filename):
    file_size = os.path.getsize(model_filename)
    print(f"✅ Modelo guardado correctamente")
    print(f"   Archivo: {model_filename}")
    print(f"   Tamaño: {file_size / 1024:.2f} KB")
else:
    print("❌ Error: el archivo no fue creado")

💾 Guardando modelo...
✅ Modelo guardado correctamente
   Archivo: heart_disease_model.joblib
   Tamaño: 2.17 KB


In [27]:
# Prueba de carga del modelo
print("🔄 Probando carga del modelo...\n")

# Cargar el modelo
loaded_pipeline = joblib.load(model_filename)

print("✅ Modelo cargado correctamente")
print(f"   Tipo: {type(loaded_pipeline)}")
print(f"   Pasos: {loaded_pipeline.steps}")

# Verificar que funciona
print("\n🧪 Verificando que el modelo cargado funciona...")

test_patient = X_test.iloc[0:1]
prediction = loaded_pipeline.predict(test_patient)

print(f"   Predicción de prueba: {prediction[0]}")
print("✅ El modelo cargado funciona correctamente")

🔄 Probando carga del modelo...

✅ Modelo cargado correctamente
   Tipo: <class 'sklearn.pipeline.Pipeline'>
   Pasos: [('scaler', StandardScaler()), ('classifier', LogisticRegression(max_iter=1000, random_state=42))]

🧪 Verificando que el modelo cargado funciona...
   Predicción de prueba: 0
✅ El modelo cargado funciona correctamente


### ✅ Checkpoint 1.7 - PARTE 1 COMPLETA

**Deberías tener:**
- ✓ Archivo `heart_disease_model.joblib` creado (tamaño ~2-5 KB)
- ✓ Archivo `heart_model_wrapper.py` creado
- ✓ Modelo con accuracy > 75%
- ✓ Wrapper funcionando correctamente
- ✓ Modelo puede cargarse y predecir

**Archivos generados hasta ahora:**
```
Tema_07/
├── heart_disease_model.joblib  ✅
├── heart_model_wrapper.py      ✅
└── Ejercicio_Practico_Despliegue_ML.ipynb (este archivo)
```

<i style="color: red">RETO: Comparte tu modelo con un compañero y pídele que lo pruebe usando el código fuente necesario para cargar dicho modelo y hacer predicciones. Es necesario usar el wrapper para garantizar que la lógica de inferencia se mantiene consistente.</i>

🎉 **¡PARTE 1 COMPLETADA!** Tiempo estimado: 30 minutos

# Parte 2: API REST con FastAPI (30 min)

## ¿Por qué necesitamos una API?

Una API REST permite que nuestro modelo sea accesible desde cualquier cliente:
- Aplicaciones web (JavaScript/React/Angular)
- Aplicaciones móviles (iOS/Android)
- Otros servicios backend
- Scripts de Python, R, etc.

## ¿Por qué FastAPI?

FastAPI es el framework moderno para crear APIs en Python:

✅ **Velocidad:** Uno de los frameworks más rápidos  
✅ **Validación automática:** Pydantic valida datos de entrada  
✅ **Documentación automática:** OpenAPI (Swagger UI) gratis  
✅ **Type hints:** Aprovecha las anotaciones de tipo de Python  
✅ **Async nativo:** Soporte para operaciones asíncronas  
✅ **Fácil de aprender:** Sintaxis muy intuitiva  

## Estructura de la API

Nuestra API tendrá los siguientes endpoints:

| Método | Endpoint | Descripción |
|--------|----------|-------------|
| GET | `/` | Información de la API |
| GET | `/health` | Health check |
| GET | `/info` | Información del modelo |
| POST | `/predict` | Predicción individual |
| POST | `/predict/batch` | Predicción batch |
| GET | `/metrics` | Métricas de Prometheus (Parte 4) |

## 📝 Paso 2.1: Crear el Archivo de la API

**Tarea:** Crear el archivo `api_heart.py` con FastAPI.

**Instrucciones:**
1. Ejecuta la siguiente celda para crear el archivo completo
2. Revisa el código generado
3. Entiende cada endpoint

### ✅ Checkpoint 2.1

**Deberías tener:**
- ✓ Archivo `api_heart.py` creado (~200 líneas)
- ✓ Imports de FastAPI, Pydantic, joblib
- ✓ Schemas definidos: PatientData, PredictionResponse
- ✓ 6 endpoints implementados

## 📝 Paso 2.2: Ejecutar la API Localmente

**Tarea:** Iniciar el servidor FastAPI con Uvicorn.

**Instrucciones:**
1. Abre una **terminal** (PowerShell o CMD en Windows, bash en Linux/Mac)
2. Navega a la carpeta `Tema_07`
3. Ejecuta el siguiente comando:

```bash
uvicorn api_heart:app --reload --host 0.0.0.0 --port 8000
```

**Explicación de los parámetros:**
- `api_heart:app` → Módulo:aplicación (archivo api_heart.py, objeto app)
- `--reload` → Recarga automática al modificar el código
- `--host 0.0.0.0` → Escucha en todas las interfaces de red
- `--port 8000` → Puerto donde corre la API

**Deberías ver:**
```
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [xxxxx]
📥 Cargando modelo...
✅ Modelo cargado correctamente
INFO:     Started server process [xxxxx]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
```

⚠️ **IMPORTANTE:** Deja la terminal abierta mientras ejecutas las siguientes celdas. No cierres el servidor.

## 📝 Paso 2.3: Explorar la Documentación Automática

**Tarea:** Abrir la interfaz interactiva de Swagger UI.

**Instrucciones:**
1. Abre tu navegador web
2. Ve a: **http://localhost:8000/docs**
3. Deberías ver la interfaz interactiva de Swagger UI

**En la interfaz verás:**
- ✅ Lista de todos los endpoints
- ✅ Schemas de datos (PatientData, PredictionResponse)
- ✅ Botón "Try it out" para probar cada endpoint
- ✅ Ejemplos de request y response

**Prueba interactiva:**
1. Haz clic en el endpoint `POST /predict`
2. Clic en "Try it out"
3. El ejemplo ya está precargado, solo haz clic en "Execute"
4. Observa la respuesta con la predicción

**Alternativa:** También puedes ir a **http://localhost:8000/redoc** para ver documentación en formato ReDoc (más bonito pero menos interactivo).

## 📝 Paso 2.4: Probar los Endpoints con Python

**Tarea:** Probar la API usando la librería `requests` desde este notebook.

**Instrucciones:**
1. Asegúrate de que el servidor esté corriendo (paso anterior)
2. Ejecuta las siguientes celdas para probar cada endpoint

In [28]:
import requests
import json

# URL base de la API
BASE_URL = "http://localhost:8000"

print("🔗 Probando conexión con la API...")
print(f"   URL: {BASE_URL}")
print("=" * 60)

🔗 Probando conexión con la API...
   URL: http://localhost:8000


In [29]:
# Prueba 1: Endpoint raíz /
print("🧪 Prueba 1: GET / (Info de la API)\n")

try:
    response = requests.get(f"{BASE_URL}/")
    print(f"Status Code: {response.status_code}")
    print(f"Response:")
    print(json.dumps(response.json(), indent=2))
    print("✅ Endpoint / funciona correctamente")
except Exception as e:
    print(f"❌ Error: {e}")
    print("⚠️ Asegúrate de que el servidor esté corriendo (uvicorn api_heart:app --reload)")

print("\n" + "=" * 60)

🧪 Prueba 1: GET / (Info de la API)

❌ Error: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: / (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión"))
⚠️ Asegúrate de que el servidor esté corriendo (uvicorn api_heart:app --reload)



In [30]:
# Prueba 2: Health check
print("🧪 Prueba 2: GET /health (Health Check)\n")

try:
    response = requests.get(f"{BASE_URL}/health")
    print(f"Status Code: {response.status_code}")
    data = response.json()
    print(f"Response:")
    print(json.dumps(data, indent=2))
    
    if data['status'] == 'healthy':
        print("✅ API está saludable y el modelo está cargado")
    else:
        print("⚠️ API responde pero hay algún problema")
except Exception as e:
    print(f"❌ Error: {e}")

print("\n" + "=" * 60)

🧪 Prueba 2: GET /health (Health Check)

❌ Error: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión"))



In [31]:
# Prueba 3: Información del modelo
print("🧪 Prueba 3: GET /info (Información del Modelo)\n")

try:
    response = requests.get(f"{BASE_URL}/info")
    print(f"Status Code: {response.status_code}")
    data = response.json()
    print(f"Response:")
    print(json.dumps(data, indent=2))
    print(f"\n✅ Modelo: {data['model_steps']}")
    print(f"✅ Características: {data['n_features']}")
except Exception as e:
    print(f"❌ Error: {e}")

print("\n" + "=" * 60)

🧪 Prueba 3: GET /info (Información del Modelo)

❌ Error: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /info (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión"))



In [32]:
# Prueba 4: Predicción individual
print("🧪 Prueba 4: POST /predict (Predicción Individual)\n")

# Datos de un paciente con NOMBRES CORRECTOS del dataset
patient_data = {
    "age": 63,
    "sex": 1,
    "chest": 3,
    "resting_blood_pressure": 145,
    "serum_cholestoral": 233,
    "fasting_blood_sugar": 1,
    "resting_electrocardiographic_results": 0,
    "maximum_heart_rate_achieved": 150,
    "exercise_induced_angina": 0,
    "oldpeak": 2.3,
    "slope": 1,
    "number_of_major_vessels": 0,
    "thal": 6
}

print("📤 Enviando datos del paciente:")
print(f"   Edad: {patient_data['age']} años")
print(f"   Sexo: {'Masculino' if patient_data['sex'] == 1 else 'Femenino'}")
print(f"   Presión arterial: {patient_data['resting_blood_pressure']} mmHg")
print(f"   Colesterol: {patient_data['serum_cholestoral']} mg/dl\n")

try:
    response = requests.post(
        f"{BASE_URL}/predict",
        json=patient_data
    )
    
    print(f"Status Code: {response.status_code}\n")
    
    if response.status_code == 200:
        result = response.json()
        print("📥 Respuesta de la API:")
        print(json.dumps(result, indent=2))
        print(f"\n🎯 RESULTADO:")
        print(f"   Predicción: {result['prediction_label']}")
        print(f"   Nivel de riesgo: {result['risk_level']}")
        print(f"   Confianza: {result['confidence']*100:.2f}%")
        print(f"   Tiempo de inferencia: {result['inference_time_ms']:.2f} ms")
        print("✅ Predicción exitosa")
    else:
        print(f"❌ Error: {response.text}")
        
except Exception as e:
    print(f"❌ Error: {e}")
    print("💡 Asegúrate de que el servidor esté corriendo")

print("\n" + "=" * 60)

🧪 Prueba 4: POST /predict (Predicción Individual)

📤 Enviando datos del paciente:
   Edad: 63 años
   Sexo: Masculino
   Presión arterial: 145 mmHg
   Colesterol: 233 mg/dl

❌ Error: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /predict (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión"))
💡 Asegúrate de que el servidor esté corriendo



In [33]:
# Prueba 5: Predicción batch
print("🧪 Prueba 5: POST /predict/batch (Predicción Batch)\n")

# Varios pacientes con NOMBRES CORRECTOS del dataset
batch_data = {
    "patients": [
        {
            "age": 45, "sex": 0, "chest": 2, "resting_blood_pressure": 120,
            "serum_cholestoral": 180, "fasting_blood_sugar": 0, 
            "resting_electrocardiographic_results": 0,
            "maximum_heart_rate_achieved": 170, "exercise_induced_angina": 0, 
            "oldpeak": 0.5, "slope": 2, "number_of_major_vessels": 0, "thal": 3
        },
        {
            "age": 67, "sex": 1, "chest": 4, "resting_blood_pressure": 160,
            "serum_cholestoral": 286, "fasting_blood_sugar": 1, 
            "resting_electrocardiographic_results": 2,
            "maximum_heart_rate_achieved": 108, "exercise_induced_angina": 1, 
            "oldpeak": 4.2, "slope": 1, "number_of_major_vessels": 3, "thal": 7
        },
        {
            "age": 52, "sex": 1, "chest": 1, "resting_blood_pressure": 130,
            "serum_cholestoral": 200, "fasting_blood_sugar": 0, 
            "resting_electrocardiographic_results": 1,
            "maximum_heart_rate_achieved": 155, "exercise_induced_angina": 0, 
            "oldpeak": 1.0, "slope": 2, "number_of_major_vessels": 1, "thal": 6
        }
    ]
}

print(f"📤 Enviando {len(batch_data['patients'])} pacientes...\n")

try:
    response = requests.post(
        f"{BASE_URL}/predict/batch",
        json=batch_data
    )
    
    print(f"Status Code: {response.status_code}\n")
    
    if response.status_code == 200:
        result = response.json()
        
        print("📥 Respuesta de la API:")
        print(f"   Total procesados: {result['count']}")
        print(f"   Tiempo total: {result['total_inference_time_ms']:.2f} ms")
        print(f"   Tiempo promedio: {result['avg_inference_time_ms']:.2f} ms/paciente\n")
        
        print("🎯 RESULTADOS:")
        for i, pred in enumerate(result['predictions'], 1):
            print(f"\n   Paciente {i}:")
            print(f"      Predicción: {pred['prediction_label']}")
            print(f"      Riesgo: {pred['risk_level']}")
            print(f"      Confianza: {pred['confidence']*100:.2f}%")
        
        print("\n✅ Predicción batch exitosa")
    else:
        print(f"❌ Error: {response.text}")
        
except Exception as e:
    print(f"❌ Error: {e}")
    print("💡 Asegúrate de que el servidor esté corriendo")

print("\n" + "=" * 60)

🧪 Prueba 5: POST /predict/batch (Predicción Batch)

📤 Enviando 3 pacientes...

❌ Error: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /predict/batch (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión"))
💡 Asegúrate de que el servidor esté corriendo



### ✅ Checkpoint 2.4 - PARTE 2 COMPLETA

**Deberías haber visto:**
- ✓ Servidor FastAPI corriendo sin errores
- ✓ Documentación interactiva en http://localhost:8000/docs
- ✓ Endpoint `/` responde con info de la API
- ✓ Endpoint `/health` muestra "status": "healthy"
- ✓ Endpoint `/info` muestra información del modelo
- ✓ Endpoint `/predict` devuelve predicción individual
- ✓ Endpoint `/predict/batch` procesa múltiples pacientes
- ✓ Tiempos de inferencia < 50 ms por predicción

**Archivos generados hasta ahora:**
```
Tema_07/
├── heart_disease_model.joblib          ✅
├── heart_model_wrapper.py              ✅
├── api_heart.py                        ✅ (NUEVO)
└── Ejercicio_Practico_Despliegue_ML.ipynb
```

**💡 Comandos útiles de curl (alternativa a Python requests):**

```bash
# GET /health
curl http://localhost:8000/health

# POST /predict
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"age": 63, "sex": 1, "chest_pain": 3, "rest_blood_pressure": 145, "cholesterol": 233, "fasting_blood_sugar": 1, "rest_ecg": 0, "max_heart_rate": 150, "exercise_angina": 0, "oldpeak": 2.3, "slope": 1, "vessels": 0, "thal": 6}'
```

🎉 **¡PARTE 2 COMPLETADA!** Tiempo estimado: 30 minutos

**Antes de continuar a la Parte 3:**
- Puedes detener el servidor con `Ctrl+C` en la terminal (lo volveremos a necesitar para Docker)

# Parte 3: Contenerización con Docker (40 min)

## ¿Por qué Docker?

Docker resuelve el problema **"funciona en mi máquina"**:

✅ **Portabilidad:** El contenedor funciona igual en cualquier máquina  
✅ **Aislamiento:** No afecta ni es afectado por otras aplicaciones  
✅ **Reproducibilidad:** Mismo entorno siempre, sin sorpresas  
✅ **Fácil despliegue:** Se despliega como una unidad atómica  
✅ **Escalabilidad:** Fácil replicar instancias  
✅ **Versionado:** Imágenes versionadas como código  

## Conceptos clave

**Imagen Docker:**
- Template inmutable que contiene:
  - Sistema operativo base (ej: Ubuntu, Alpine)
  - Dependencias y librerías
  - Código de la aplicación
  - Configuración

**Contenedor Docker:**
- Instancia en ejecución de una imagen
- Como una "mini máquina virtual" muy ligera
- Se puede iniciar, detener, eliminar

**Dockerfile:**
- Archivo de texto con instrucciones para construir la imagen
- Cada línea crea una "capa" en la imagen

## Arquitectura de nuestra imagen

```
┌────────────────────────────────────┐
│  Contenedor Heart Disease API     │
├────────────────────────────────────┤
│  Puerto 8000:8000                  │
│                                    │
│  ┌──────────────────────────────┐ │
│  │  FastAPI App (api_heart.py)  │ │
│  └──────────────────────────────┘ │
│                                    │
│  ┌──────────────────────────────┐ │
│  │  Modelo (heart_disease.joblib)│ │
│  └──────────────────────────────┘ │
│                                    │
│  ┌──────────────────────────────┐ │
│  │  Wrapper (heart_model_wrapper)│ │
│  └──────────────────────────────┘ │
│                                    │
│  ┌──────────────────────────────┐ │
│  │  Python 3.9 + Dependencies   │ │
│  └──────────────────────────────┘ │
│                                    │
│  ┌──────────────────────────────┐ │
│  │  OS: Debian Slim             │ │
│  └──────────────────────────────┘ │
└────────────────────────────────────┘
```

## 📝 Paso 3.1: Crear requirements.txt

**Tarea:** Definir todas las dependencias del proyecto con versiones fijadas.

**¿Por qué fijar versiones?**
- Reproducibilidad: mismo comportamiento siempre
- Evita incompatibilidades futuras
- Facilita debugging

**Instrucciones:**
1. Ejecuta la siguiente celda para crear el archivo
2. Observa las versiones fijadas

In [34]:
requirements_txt = """# Core ML
scikit-learn==1.0.2
joblib==1.1.0
numpy==1.22.3
pandas==1.4.2

# API Framework
fastapi==0.95.0
uvicorn[standard]==0.21.1
pydantic==1.10.7

# Monitoring (Parte 4)
prometheus-client==0.16.0
"""

# Guardar archivo
with open('requirements.txt', 'w', encoding='utf-8') as f:
    f.write(requirements_txt.strip())

print("✅ Archivo 'requirements.txt' creado correctamente\n")
print("📦 Dependencias:")
for line in requirements_txt.strip().split('\n'):
    if line and not line.startswith('#'):
        print(f"   - {line}")

✅ Archivo 'requirements.txt' creado correctamente

📦 Dependencias:
   - scikit-learn==1.0.2
   - joblib==1.1.0
   - numpy==1.22.3
   - pandas==1.4.2
   - fastapi==0.95.0
   - uvicorn[standard]==0.21.1
   - pydantic==1.10.7
   - prometheus-client==0.16.0


## 📝 Paso 3.2: Crear el Dockerfile

**Tarea:** Definir las instrucciones para construir la imagen Docker.

**Anatomía del Dockerfile:**
- `FROM`: Imagen base (Python 3.9 slim - versión ligera)
- `WORKDIR`: Directorio de trabajo dentro del contenedor
- `COPY`: Copiar archivos del host al contenedor
- `RUN`: Ejecutar comandos (ej: instalar dependencias)
- `EXPOSE`: Documentar el puerto que usa la app
- `CMD`: Comando por defecto al iniciar el contenedor

**Mejores prácticas aplicadas:**
✓ Imagen base ligera (python:3.9-slim)  
✓ Usuario no-root para seguridad  
✓ Caché de capas optimizado  
✓ Health check incorporado  
✓ Labels para metadata  

**Instrucciones:**
1. Ejecuta la siguiente celda para crear el Dockerfile
2. Lee los comentarios para entender cada instrucción

In [35]:
dockerfile_content = """# Imagen base oficial de Python 3.9 (versión slim = ligera)
FROM python:3.9-slim

# Metadata de la imagen
LABEL maintainer="tu-email@ejemplo.com"
LABEL version="1.0"
LABEL description="Heart Disease Prediction API"

# Establecer directorio de trabajo
WORKDIR /app

# Instalar dependencias del sistema (si son necesarias)
RUN apt-get update && apt-get install -y \\
    --no-install-recommends \\
    && rm -rf /var/lib/apt/lists/*

# Copiar archivo de requirements primero (para aprovechar cache de Docker)
COPY requirements.txt .

# Instalar dependencias de Python
RUN pip install --no-cache-dir --upgrade pip && \\
    pip install --no-cache-dir -r requirements.txt

# Copiar el código de la aplicación
COPY api_heart.py .
COPY heart_model_wrapper.py .
COPY heart_disease_model.joblib .

# Crear usuario no-root para seguridad
RUN useradd -m -u 1000 appuser && \\
    chown -R appuser:appuser /app

# Cambiar a usuario no-root
USER appuser

# Exponer el puerto 8000
EXPOSE 8000

# Health check: verificar que la API responde
HEALTHCHECK --interval=30s --timeout=10s --start-period=5s --retries=3 \\
    CMD python -c "import requests; requests.get('http://localhost:8000/health')"

# Comando por defecto: iniciar uvicorn
CMD ["uvicorn", "api_heart:app", "--host", "0.0.0.0", "--port", "8000"]
"""

# Guardar archivo
with open('Dockerfile', 'w', encoding='utf-8') as f:
    f.write(dockerfile_content.strip())

print("✅ Archivo 'Dockerfile' creado correctamente\n")
print("📋 Estructura del Dockerfile:")
print("   1. FROM: Python 3.9 slim")
print("   2. WORKDIR: /app")
print("   3. COPY: requirements.txt")
print("   4. RUN: pip install dependencies")
print("   5. COPY: código de la aplicación")
print("   6. RUN: crear usuario appuser")
print("   7. USER: cambiar a appuser (seguridad)")
print("   8. EXPOSE: puerto 8000")
print("   9. HEALTHCHECK: verificar /health cada 30s")
print("  10. CMD: uvicorn api_heart:app")

✅ Archivo 'Dockerfile' creado correctamente

📋 Estructura del Dockerfile:
   1. FROM: Python 3.9 slim
   2. WORKDIR: /app
   3. COPY: requirements.txt
   4. RUN: pip install dependencies
   5. COPY: código de la aplicación
   6. RUN: crear usuario appuser
   7. USER: cambiar a appuser (seguridad)
   8. EXPOSE: puerto 8000
   9. HEALTHCHECK: verificar /health cada 30s
  10. CMD: uvicorn api_heart:app


## 📝 Paso 3.3: Crear .dockerignore

**Tarea:** Definir qué archivos NO copiar al contenedor.

**¿Por qué .dockerignore?**
- Reduce tamaño de la imagen
- Acelera el build
- Evita copiar archivos innecesarios o sensibles

**Instrucciones:**
1. Ejecuta la celda para crear el archivo

In [36]:
dockerignore_content = """# Python
__pycache__/
*.py[cod]
*$py.class
*.so
.Python
env/
venv/
ENV/
.venv

# Jupyter
.ipynb_checkpoints/
*.ipynb

# IDEs
.vscode/
.idea/
*.swp
*.swo

# Git
.git/
.gitignore

# Docker
Dockerfile
.dockerignore
docker-compose.yml

# Tests y documentación
tests/
docs/
*.md
README.md

# OS
.DS_Store
Thumbs.db

# Logs
*.log

# Archivos temporales
tmp/
temp/
"""

# Guardar archivo
with open('.dockerignore', 'w', encoding='utf-8') as f:
    f.write(dockerignore_content.strip())

print("✅ Archivo '.dockerignore' creado correctamente\n")
print("🚫 Archivos ignorados:")
print("   - Notebooks (.ipynb)")
print("   - Python cache (__pycache__)")
print("   - Entornos virtuales (venv/)")
print("   - IDEs (.vscode/, .idea/)")
print("   - Git (.git/)")
print("   - Logs y archivos temporales")

✅ Archivo '.dockerignore' creado correctamente

🚫 Archivos ignorados:
   - Notebooks (.ipynb)
   - Python cache (__pycache__)
   - Entornos virtuales (venv/)
   - IDEs (.vscode/, .idea/)
   - Git (.git/)
   - Logs y archivos temporales


### ✅ Checkpoint 3.3

**Deberías tener:**
- ✓ `requirements.txt` con dependencias fijadas
- ✓ `Dockerfile` con todas las instrucciones
- ✓ `.dockerignore` configurado

## 📝 Paso 3.4: Construir la Imagen Docker

**Tarea:** Construir la imagen Docker a partir del Dockerfile.

**Instrucciones:**

**1. Abre una terminal en la carpeta `Tema_07`**

**2. Ejecuta el siguiente comando:**

```bash
docker build -t heart-api:v1.0 .
```

**Explicación del comando:**
- `docker build`: construir una imagen
- `-t heart-api:v1.0`: tag (nombre:versión) de la imagen
- `.`: contexto de build (carpeta actual)

**Proceso de build:**
El build pasará por múltiples pasos (uno por instrucción del Dockerfile):

```
[1/10] FROM python:3.9-slim
[2/10] WORKDIR /app
[3/10] COPY requirements.txt
[4/10] RUN pip install...
... (más pasos)
```

**Tiempo estimado:** 2-5 minutos en el primer build (después es más rápido por caché)

**Deberías ver al final:**
```
Successfully built abc123def456
Successfully tagged heart-api:v1.0
```

**💡 Comandos útiles mientras esperas:**

```bash
# Ver el progreso en tiempo real
docker build -t heart-api:v1.0 . --progress=plain

# Build sin cache (si hay problemas)
docker build -t heart-api:v1.0 . --no-cache
```


**3. Verificar que la imagen fue creada:**

```bash
docker images | grep heart-api
```

Deberías ver algo como:
```
heart-api    v1.0    abc123def456   2 minutes ago   500MB

```

## 📝 Paso 3.5: Ejecutar el Contenedor

**Tarea:** Iniciar un contenedor a partir de la imagen construida.

**Instrucciones:**

**1. Ejecuta el siguiente comando en la terminal:**

```bash
docker run -d -p 8000:8000 --name heart-api-container heart-api:v1.0
```

**Explicación del comando:**
- `docker run`: crear y ejecutar un contenedor
- `-d`: modo detached (corre en segundo plano)
- `-p 8000:8000`: mapeo de puertos (host:contenedor)
- `--name heart-api-container`: nombre del contenedor
- `heart-api:v1.0`: imagen a usar

**Deberías ver un hash largo (ID del contenedor):**
```
abc123def456789...
```

**2. Verificar que el contenedor está corriendo:**

```bash
docker ps
```

Deberías ver algo como:
```
CONTAINER ID   IMAGE            COMMAND                  STATUS         PORTS
abc123def456   heart-api:v1.0   "uvicorn api_heart:a…"   Up 10 seconds  0.0.0.0:8000->8000/tcp
```

**3. Ver los logs del contenedor:**

```bash
docker logs heart-api-container
```

Deberías ver:
```
INFO:     Started server process [1]
INFO:     Waiting for application startup.
📥 Cargando modelo...
✅ Modelo cargado correctamente
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000
```

**💡 Comandos útiles:**

```bash
# Logs en tiempo real
docker logs -f heart-api-container

# Detener el contenedor
docker stop heart-api-container

# Reiniciar el contenedor
docker start heart-api-container

# Eliminar el contenedor
docker rm -f heart-api-container

# Ver estadísticas de recursos
docker stats heart-api-container

# Ejecutar comando dentro del contenedor
docker exec -it heart-api-container /bin/bash
```

## 📝 Paso 3.6: Probar la API en el Contenedor

**Tarea:** Verificar que la API funciona correctamente dentro del contenedor.

**Instrucciones:**
1. Ejecuta las siguientes celdas para probar los endpoints
2. Deberían funcionar exactamente igual que antes (Parte 2)

In [37]:
import requests
import json

print("🐳 Probando API en contenedor Docker...\n")
print("="*60)

# URL de la API (mismo puerto mapeado)
BASE_URL = "http://localhost:8000"

# Prueba 1: Health check
print("\n🧪 Health Check:")
try:
    response = requests.get(f"{BASE_URL}/health", timeout=5)
    print(f"✅ Status: {response.status_code}")
    print(f"   {json.dumps(response.json(), indent=2)}")
except Exception as e:
    print(f"❌ Error: {e}")
    print("💡 Asegúrate de que el contenedor esté corriendo: docker ps")

print("\n" + "="*60)

🐳 Probando API en contenedor Docker...


🧪 Health Check:
❌ Error: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión"))
💡 Asegúrate de que el contenedor esté corriendo: docker ps



In [38]:
# Prueba 2: Predicción
print("\n🧪 Predicción en contenedor:")

# Datos con NOMBRES CORRECTOS del dataset
patient_data = {
    "age": 58, "sex": 1, "chest": 2, "resting_blood_pressure": 140,
    "serum_cholestoral": 250, "fasting_blood_sugar": 1, 
    "resting_electrocardiographic_results": 1,
    "maximum_heart_rate_achieved": 145, "exercise_induced_angina": 0, 
    "oldpeak": 1.5, "slope": 2, "number_of_major_vessels": 1, "thal": 6
}

try:
    response = requests.post(
        f"{BASE_URL}/predict",
        json=patient_data,
        timeout=5
    )
    
    if response.status_code == 200:
        result = response.json()
        print(f"✅ Predicción exitosa:")
        print(f"   Resultado: {result['prediction_label']}")
        print(f"   Riesgo: {result['risk_level']}")
        print(f"   Confianza: {result['confidence']*100:.2f}%")
        print(f"   Tiempo: {result['inference_time_ms']:.2f} ms")
    else:
        print(f"❌ Error: {response.status_code}")
        print(response.text)
        
except Exception as e:
    print(f"❌ Error: {e}")

print("\n" + "="*60)

print("\n🎉 ¡La API funciona correctamente dentro del contenedor Docker!")


🧪 Predicción en contenedor:
❌ Error: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /predict (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión"))


🎉 ¡La API funciona correctamente dentro del contenedor Docker!


### ✅ Checkpoint 3.6 - PARTE 3 COMPLETA

**Deberías tener:**
- ✓ Imagen Docker `heart-api:v1.0` construida (~500-600 MB)
- ✓ Contenedor `heart-api-container` corriendo
- ✓ API accesible en http://localhost:8000
- ✓ Health check funcionando
- ✓ Predicciones funcionando dentro del contenedor
- ✓ Logs visibles con `docker logs`

**Archivos generados:**
```
Tema_07/
├── heart_disease_model.joblib          ✅
├── heart_model_wrapper.py              ✅
├── api_heart.py                        ✅
├── requirements.txt                    ✅ (NUEVO)
├── Dockerfile                          ✅ (NUEVO)
├── .dockerignore                       ✅ (NUEVO)
└── Ejercicio_Practico_Despliegue_ML.ipynb
```

**💡 Ventajas de Docker demostradas:**

✅ **Portabilidad:** La imagen funciona en cualquier máquina con Docker  
✅ **Aislamiento:** El contenedor no afecta tu sistema local  
✅ **Reproducibilidad:** Siempre mismo entorno  
✅ **Facilidad de despliegue:** Un solo comando (`docker run`)  
✅ **Escalabilidad:** Puedes correr múltiples contenedores  

**📦 Comandos de resumen:**

```bash
# Construir imagen
docker build -t heart-api:v1.0 .

# Ejecutar contenedor
docker run -d -p 8000:8000 --name heart-api-container heart-api:v1.0

# Ver logs
docker logs heart-api-container

# Detener
docker stop heart-api-container

# Eliminar
docker rm heart-api-container

# Ver imágenes
docker images

# Eliminar imagen
docker rmi heart-api:v1.0
```

🎉 **¡PARTE 3 COMPLETADA!** Tiempo estimado: 40 minutos

**Próximo paso:** Parte 4 - Monitoreo con Prometheus (opcional, 20 min)

# Parte 4: Monitoreo Básico con Prometheus (20 min) - OPCIONAL

⚠️ **NOTA:** Esta parte es **opcional** y más avanzada. Si el tiempo de clase es limitado, puedes omitirla. Las Partes 1-3 ya conforman un sistema completo de producción.

## ¿Por qué monitorear?

En producción, necesitamos saber:
- ¿La API está respondiendo?
- ¿Cuántas predicciones se están haciendo?
- ¿Cuál es el tiempo de respuesta?
- ¿Hay errores?

## Prometheus + Grafana

**Prometheus:** Sistema de monitoreo que recolecta métricas  
**Grafana:** Herramienta de visualización de métricas con dashboards  

**Arquitectura:**

```
┌──────────────┐
│   Grafana    │  ← Dashboard visual
│   :3000      │
└──────┬───────┘
       │ consulta
       ▼
┌──────────────┐
│  Prometheus  │  ← Almacena métricas
│   :9090      │
└──────┬───────┘
       │ scrape (cada 15s)
       ▼
┌──────────────┐
│  FastAPI     │  ← Expone /metrics
│   :8000      │
└──────────────┘
```

## 📝 Paso 4.1: Instrumentar la API con Métricas

**Tarea:** Añadir métricas de Prometheus a la API.

**Instrucciones:**
1. La API ya tiene instalado `prometheus-client` (está en requirements.txt)
2. Ejecuta la siguiente celda para crear una versión mejorada de la API con métricas

In [39]:
# Código de api_heart.py CON MÉTRICAS DE PROMETHEUS
# Este código reemplaza el api_heart.py anterior

api_with_metrics = '''
from fastapi import FastAPI, HTTPException, Response
from pydantic import BaseModel, Field
from typing import List, Dict
import joblib
import pandas as pd
from heart_model_wrapper import HeartDiseaseWrapper
import time
from datetime import datetime

# NUEVO: Importar Prometheus
from prometheus_client import Counter, Histogram, Gauge, generate_latest, CONTENT_TYPE_LATEST

# Inicializar FastAPI
app = FastAPI(
    title="Heart Disease Prediction API",
    description="API para predecir enfermedad cardíaca usando Machine Learning",
    version="1.0.0"
)

# Variables globales
model = None
wrapper = None
model_loaded_time = None

# NUEVO: Métricas de Prometheus
PREDICTIONS_TOTAL = Counter(
    'predictions_total', 
    'Número total de predicciones realizadas',
    ['endpoint']
)

PREDICTION_DURATION = Histogram(
    'prediction_duration_seconds',
    'Duración de las predicciones en segundos',
    ['endpoint']
)

ACTIVE_PREDICTIONS = Gauge(
    'active_predictions',
    'Número de predicciones activas en este momento'
)

PREDICTION_ERRORS = Counter(
    'prediction_errors_total',
    'Número total de errores en predicciones'
)

# Cargar modelo al iniciar
@app.on_event("startup")
async def load_model():
    global model, wrapper, model_loaded_time
    try:
        print("📥 Cargando modelo...")
        model = joblib.load("heart_disease_model.joblib")
        wrapper = HeartDiseaseWrapper(model)
        model_loaded_time = datetime.now()
        print("✅ Modelo cargado correctamente")
    except Exception as e:
        print(f"❌ Error cargando modelo: {e}")
        raise

# Schemas
class PatientData(BaseModel):
    age: int = Field(..., ge=1, le=120)
    sex: int = Field(..., ge=0, le=1)
    chest_pain: int = Field(..., ge=1, le=4)
    rest_blood_pressure: int = Field(..., ge=50, le=250)
    cholesterol: int = Field(..., ge=100, le=600)
    fasting_blood_sugar: int = Field(..., ge=0, le=1)
    rest_ecg: int = Field(..., ge=0, le=2)
    max_heart_rate: int = Field(..., ge=60, le=220)
    exercise_angina: int = Field(..., ge=0, le=1)
    oldpeak: float = Field(..., ge=0, le=10)
    slope: int = Field(..., ge=1, le=3)
    vessels: int = Field(..., ge=0, le=3)
    thal: int = Field(..., ge=3, le=7)

class PredictionResponse(BaseModel):
    prediction: int
    prediction_label: str
    confidence: float
    probabilities: Dict[str, float]
    risk_level: str
    inference_time_ms: float

class BatchPredictionRequest(BaseModel):
    patients: List[PatientData]

# Endpoints
@app.get("/")
async def root():
    return {
        "message": "Heart Disease Prediction API",
        "version": "1.0.0",
        "endpoints": {
            "docs": "/docs",
            "health": "/health",
            "info": "/info",
            "predict": "/predict",
            "predict_batch": "/predict/batch",
            "metrics": "/metrics"
        }
    }

@app.get("/health")
async def health_check():
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    
    return {
        "status": "healthy",
        "model_loaded": model is not None,
        "model_loaded_at": model_loaded_time.isoformat() if model_loaded_time else None,
        "timestamp": datetime.now().isoformat()
    }

@app.get("/info")
async def model_info():
    if model is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    
    return {
        "model_type": str(type(model)),
        "model_steps": [step[0] for step in model.steps],
        "features": wrapper.feature_names,
        "n_features": len(wrapper.feature_names),
        "loaded_at": model_loaded_time.isoformat() if model_loaded_time else None
    }

@app.post("/predict", response_model=PredictionResponse)
async def predict(patient: PatientData):
    if wrapper is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    
    # NUEVO: Incrementar métricas
    ACTIVE_PREDICTIONS.inc()
    
    try:
        # Medir tiempo
        start_time = time.time()
        
        patient_dict = patient.dict()
        result = wrapper.predict(patient_dict)
        
        inference_time = time.time() - start_time
        result['inference_time_ms'] = round(inference_time * 1000, 2)
        
        # NUEVO: Registrar métricas
        PREDICTIONS_TOTAL.labels(endpoint='predict').inc()
        PREDICTION_DURATION.labels(endpoint='predict').observe(inference_time)
        
        return result
        
    except Exception as e:
        PREDICTION_ERRORS.inc()
        raise HTTPException(status_code=500, detail=f"Prediction error: {str(e)}")
    finally:
        ACTIVE_PREDICTIONS.dec()

@app.post("/predict/batch")
async def predict_batch(request: BatchPredictionRequest):
    if wrapper is None:
        raise HTTPException(status_code=503, detail="Model not loaded")
    
    ACTIVE_PREDICTIONS.inc()
    
    try:
        start_time = time.time()
        
        patients_list = [patient.dict() for patient in request.patients]
        results = wrapper.predict_batch(patients_list)
        
        inference_time = time.time() - start_time
        
        # NUEVO: Registrar métricas
        PREDICTIONS_TOTAL.labels(endpoint='predict_batch').inc(len(results))
        PREDICTION_DURATION.labels(endpoint='predict_batch').observe(inference_time)
        
        return {
            "predictions": results,
            "count": len(results),
            "total_inference_time_ms": round(inference_time * 1000, 2),
            "avg_inference_time_ms": round(inference_time / len(results) * 1000, 2)
        }
        
    except Exception as e:
        PREDICTION_ERRORS.inc()
        raise HTTPException(status_code=500, detail=f"Batch prediction error: {str(e)}")
    finally:
        ACTIVE_PREDICTIONS.dec()

# NUEVO: Endpoint de métricas para Prometheus
@app.get("/metrics")
async def metrics():
    """Expone métricas en formato Prometheus."""
    return Response(content=generate_latest(), media_type=CONTENT_TYPE_LATEST)

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

# Guardar el archivo actualizado
with open('api_heart.py', 'w', encoding='utf-8') as f:
    f.write(api_with_metrics.strip())

print("✅ Archivo 'api_heart.py' actualizado con métricas de Prometheus\n")
print("📊 Métricas añadidas:")
print("   - predictions_total: contador de predicciones")
print("   - prediction_duration_seconds: histograma de tiempos")
print("   - active_predictions: gauge de predicciones activas")
print("   - prediction_errors_total: contador de errores")
print("\n💡 Nuevo endpoint: GET /metrics")

✅ Archivo 'api_heart.py' actualizado con métricas de Prometheus

📊 Métricas añadidas:
   - predictions_total: contador de predicciones
   - prediction_duration_seconds: histograma de tiempos
   - active_predictions: gauge de predicciones activas
   - prediction_errors_total: contador de errores

💡 Nuevo endpoint: GET /metrics


## 📝 Paso 4.2: Probar el Endpoint de Métricas

**Tarea:** Verificar que las métricas se están exponiendo correctamente.

**Instrucciones:**

**1. Reiniciar la API con el código actualizado:**

En terminal (si estaba corriendo, deténla con Ctrl+C primero):

```bash
uvicorn api_heart:app --reload
```

**2. Hacer algunas predicciones para generar métricas:**

Ejecuta la siguiente celda para hacer varias predicciones:

In [40]:
import requests
import time

BASE_URL = "http://localhost:8000"

print("🔄 Generando tráfico para métricas...\n")

# Hacer varias predicciones con NOMBRES CORRECTOS del dataset
patient_data = {
    "age": 55, "sex": 1, "chest": 2, "resting_blood_pressure": 130,
    "serum_cholestoral": 220, "fasting_blood_sugar": 0, 
    "resting_electrocardiographic_results": 1,
    "maximum_heart_rate_achieved": 155, "exercise_induced_angina": 0, 
    "oldpeak": 1.2, "slope": 2, "number_of_major_vessels": 1, "thal": 6
}

# Hacer 10 predicciones
for i in range(10):
    try:
        response = requests.post(f"{BASE_URL}/predict", json=patient_data, timeout=5)
        if response.status_code == 200:
            print(f"✅ Predicción {i+1}/10 exitosa")
        time.sleep(0.2)  # Pequeña pausa
    except Exception as e:
        print(f"❌ Error en predicción {i+1}: {e}")

print("\n✅ Tráfico generado. Ahora hay métricas disponibles.")

🔄 Generando tráfico para métricas...

❌ Error en predicción 1: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /predict (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión"))
❌ Error en predicción 2: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /predict (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión"))
❌ Error en predicción 3: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /predict (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No se puede establecer 

In [41]:
# Consultar endpoint /metrics
print("📊 Consultando métricas de Prometheus...\n")

try:
    response = requests.get(f"{BASE_URL}/metrics", timeout=5)
    
    if response.status_code == 200:
        metrics_text = response.text
        
        print("✅ Métricas obtenidas correctamente\n")
        print("="*60)
        print("Formato Prometheus (extracto):\n")
        
        # Mostrar solo las líneas relevantes
        for line in metrics_text.split('\n'):
            if any(keyword in line for keyword in [
                'predictions_total',
                'prediction_duration',
                'active_predictions',
                'prediction_errors'
            ]):
                print(line)
        
        print("="*60)
        print("\n💡 Puedes ver las métricas completas en:")
        print("   http://localhost:8000/metrics")
        
        # Parsear algunas métricas clave
        print("\n📈 Resumen de métricas:")
        for line in metrics_text.split('\n'):
            if line.startswith('predictions_total{'):
                print(f"   {line}")
            elif line.startswith('prediction_duration_seconds_count'):
                print(f"   {line}")
                
    else:
        print(f"❌ Error: {response.status_code}")
        
except Exception as e:
    print(f"❌ Error: {e}")
    print("💡 Asegúrate de que la API esté corriendo con el código actualizado")

📊 Consultando métricas de Prometheus...

❌ Error: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /metrics (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No se puede establecer una conexión ya que el equipo de destino denegó expresamente dicha conexión"))
💡 Asegúrate de que la API esté corriendo con el código actualizado


### ✅ Checkpoint 4.2 - PARTE 4 COMPLETA

**Deberías tener:**
- ✓ API actualizada con métricas de Prometheus
- ✓ Endpoint `/metrics` funcionando
- ✓ Métricas incrementándose con cada predicción:
  - `predictions_total`: contador total
  - `prediction_duration_seconds`: histograma de tiempos
  - `active_predictions`: gauge de requests activos
  - `prediction_errors_total`: contador de errores

**Formato de métricas Prometheus:**

```
# HELP predictions_total Número total de predicciones realizadas
# TYPE predictions_total counter
predictions_total{endpoint="predict"} 10.0

# HELP prediction_duration_seconds Duración de las predicciones en segundos
# TYPE prediction_duration_seconds histogram
prediction_duration_seconds_bucket{endpoint="predict",le="0.005"} 8.0
prediction_duration_seconds_bucket{endpoint="predict",le="0.01"} 10.0
prediction_duration_seconds_count{endpoint="predict"} 10.0
prediction_duration_seconds_sum{endpoint="predict"} 0.0234
```

**🚀 Próximos pasos (fuera del alcance de esta clase):**

Si quieres ir más allá, puedes:

1. **Configurar Prometheus para scrapear métricas:**
   - Instalar Prometheus
   - Configurar scrape de http://localhost:8000/metrics

2. **Configurar Grafana:**
   - Instalar Grafana
   - Conectar con Prometheus como datasource
   - Crear dashboards personalizados

3. **Usar docker-compose para orquestar:**
   ```yaml
   services:
     api:
       build: .
       ports:
         - "8000:8000"
     prometheus:
       image: prom/prometheus
       ports:
         - "9090:9090"
     grafana:
       image: grafana/grafana
       ports:
         - "3000:3000"
   ```

**📚 Recursos para profundizar:**
- [Prometheus Docs](https://prometheus.io/docs/)
- [Grafana Docs](https://grafana.com/docs/)
- [prometheus_client para Python](https://github.com/prometheus/client_python)

🎉 **¡PARTE 4 COMPLETADA!** Tiempo estimado: 20 minutos

# Conclusiones y reflexión final

### ✅ Lo que has construido

**1. Sistema de ML completo:**
- Modelo entrenado y validado (Logistic Regression)
- Wrapper que encapsula toda la lógica de inferencia
- API REST profesional con FastAPI
- Contenedor Docker listo para producción

**2. Habilidades adquiridas:**
- Serialización de modelos con Joblib
- Diseño de interfaces limpias con wrappers
- Desarrollo de APIs REST
- Contenerización con Docker
- Instrumentación de métricas
- Mejores prácticas de MLOps

**3. Archivos generados:**
```
Tema_07/
├── Ejercicio_Practico_Despliegue_ML.ipynb  ← Este cuaderno
├── heart_disease_model.joblib               ← Modelo serializado
├── heart_model_wrapper.py                   ← Wrapper del modelo
├── api_heart.py                             ← API FastAPI con métricas
├── requirements.txt                          ← Dependencias
├── Dockerfile                                ← Definición de imagen
└── .dockerignore                             ← Archivos ignorados
```

## 🚀 Del Notebook a Producción: El Viaje Completo

```
Experimentación (Notebook)
        │
        ├─ Entrenar modelo
        ├─ Evaluar métricas
        └─ Iterar
        │
        ▼
Empaquetado
        │
        ├─ Serializar modelo (Joblib)
        ├─ Crear wrapper
        └─ Definir dependencias
        │
        ▼
API REST
        │
        ├─ Endpoints HTTP
        ├─ Validación automática (Pydantic)
        └─ Documentación (OpenAPI)
        │
        ▼
Contenerización (Docker)
        │
        ├─ Imagen reproducible
        ├─ Aislamiento
        └─ Portabilidad
        │
        ▼
Monitoreo
        │
        ├─ Métricas (Prometheus)
        ├─ Logs
        └─ Health checks
        │
        ▼
PRODUCCIÓN 
```

## 💡 Lecciones clave

**1. El modelo es solo el 10% del sistema**
- El 90% restante es infraestructura, APIs, monitoreo, etc.
- Un buen modelo mal desplegado no genera valor

**2. La reproducibilidad es fundamental**
- Requirements con versiones fijadas
- Docker para entornos consistentes
- Git para control de versiones

**3. Las APIs REST son el estándar**
- Facilitan integración con cualquier cliente
- FastAPI hace todo más fácil y rápido

**4. Docker cambia las reglas del juego**
- "Funciona en mi máquina" → "Funciona en todas partes"
- Facilita despliegue y escalamiento

**5. El monitoreo no es opcional**
- Necesitas saber cómo está funcionando tu modelo
- Las métricas te alertan de problemas antes que los usuarios

## 🔄 Siguientes pasos sugeridos

**Para mejorar este sistema:**

1. **Testing:**
   - Añadir tests unitarios (pytest)
   - Tests de integración de la API
   - Tests de carga (locust, k6)

2. **CI/CD:**
   - GitHub Actions para tests automáticos
   - Build automático de imágenes Docker
   - Despliegue automático a staging/producción

3. **Seguridad:**
   - Autenticación (API keys, JWT)
   - HTTPS/TLS
   - Rate limiting
   - Input sanitization

4. **Escalabilidad:**
   - Load balancer (nginx)
   - Múltiples réplicas del contenedor
   - Kubernetes para orquestación

5. **MLOps avanzado:**
   - Model registry (MLflow)
   - Feature store
   - A/B testing
   - Continuous training
   - Drift detection

6. **Observabilidad:**
   - Logs centralizados (ELK stack)
   - Tracing distribuido (Jaeger)
   - Alertas (Alertmanager)

## 📚 Recursos recomendados

**Libros:**
- "Designing Machine Learning Systems" - Chip Huyen
- "Building Machine Learning Powered Applications" - Emmanuel Ameisen
- "ML Engineering" - Andriy Burkov

**Cursos:**
- [Made With ML](https://madewithml.com/) - MLOps completo y gratuito
- [Full Stack Deep Learning](https://fullstackdeeplearning.com/)

**Documentación:**
- [FastAPI Docs](https://fastapi.tiangolo.com/)
- [Docker Docs](https://docs.docker.com/)
- [scikit-learn Docs](https://scikit-learn.org/)
- [Prometheus Docs](https://prometheus.io/docs/)

**Comunidades:**
- r/MachineLearning
- MLOps Community (mlops.community)
- FastAPI Discord

## 🎯 Reflexión final

> "El valor de un modelo de ML no está en su accuracy, sino en el valor de negocio que genera cuando está correctamente desplegado y mantenido en producción."

Este ejercicio te ha mostrado que desplegar ML en producción requiere:
- **Conocimiento técnico:** APIs, Docker, infraestructura
- **Mejores prácticas:** Testing, monitoreo, documentación
- **Pensamiento sistémico:** No es solo el modelo, es todo el pipeline

**El camino de MLOps es continuo:**
- Empieza simple (como hicimos hoy)
- Itera y mejora
- Automatiza todo lo que puedas
- Monitorea y aprende

¡Ahora tienes las bases para llevar tus modelos de ML a producción de forma profesional! 🚀

## 👨‍🏫 Agradecimientos

Gracias por completar este ejercicio práctico. Espero que te haya sido útil y que apliques estos conocimientos en tus proyectos reales.


# Troubleshooting - Problemas Comunes y Soluciones

Esta sección te ayudará a resolver los problemas más comunes que puedes encontrar durante el ejercicio.

## 🐍 Problemas con Python y Dependencias

### Error: "ModuleNotFoundError: No module named 'sklearn'"

**Causa:** scikit-learn no está instalado

**Solución:**
```bash
pip install scikit-learn==1.0.2
# o
pip install -r requirements.txt
```

### Error: "ImportError: cannot import name 'BaseModel' from 'pydantic'"

**Causa:** Versión incorrecta de Pydantic

**Solución:**
```bash
pip install pydantic==1.10.7
```

### Error: Versiones incompatibles entre dependencias

**Solución:**
```bash
# Desinstalar todo y reinstalar limpio
pip uninstall scikit-learn pandas numpy fastapi uvicorn -y
pip install -r requirements.txt
```

## 🌐 Problemas con la API (uvicorn)

### Error: "Address already in use" o "Port 8000 is already allocated"

**Causa:** El puerto 8000 ya está siendo usado por otro proceso

**Soluciones:**

**Opción 1: Usar otro puerto**
```bash
uvicorn api_heart:app --reload --port 8001
```

**Opción 2: Detener el proceso que usa el puerto (Windows)**
```powershell
# Ver qué proceso usa el puerto 8000
netstat -ano | findstr :8000

# Matar el proceso (reemplaza PID con el número que obtuviste)
taskkill /PID <PID> /F
```

**Opción 3: Detener el proceso (Linux/Mac)**
```bash
# Ver qué proceso usa el puerto
lsof -i :8000

# Matar el proceso
kill -9 <PID>
```

### Error: "Model not loaded" al hacer requests

**Causa:** El modelo no se cargó al iniciar la API

**Solución:**
1. Verifica que el archivo `heart_disease_model.joblib` existe en el directorio
2. Revisa los logs de uvicorn para ver el error exacto
3. Asegúrate de que `heart_model_wrapper.py` está en el mismo directorio

### Error: "Validation error" al hacer POST /predict

**Causa:** Los datos enviados no cumplen con el schema

**Solución:**
- Verifica que todos los campos requeridos están presentes
- Verifica que los tipos de datos son correctos (int, float)
- Verifica que los valores están dentro de los rangos permitidos
- Ejemplo de request correcto:
```python
{
    "age": 63,  # int, 1-120
    "sex": 1,   # int, 0 o 1
    "chest_pain": 3,  # int, 1-4
    # ... (todos los 13 campos)
}
```

## 🐳 Problemas con Docker

### Error: "docker: command not found"

**Causa:** Docker no está instalado o no está en el PATH

**Solución:**
1. Instala Docker Desktop (Windows/Mac): https://www.docker.com/products/docker-desktop
2. En Linux: instala Docker Engine
3. Verifica instalación: `docker --version`

### Error: "Cannot connect to the Docker daemon"

**Causa:** Docker Desktop no está corriendo

**Solución:**
1. Abre Docker Desktop
2. Espera a que inicie completamente (icono en la barra de tareas)
3. Verifica con: `docker ps`

### Error: "Permission denied" al ejecutar comandos Docker (Linux)

**Causa:** Usuario no tiene permisos para usar Docker

**Solución:**
```bash
# Añadir usuario al grupo docker
sudo usermod -aG docker $USER

# Cerrar sesión y volver a entrar, o ejecutar:
newgrp docker

# Verificar
docker ps
```

### Error: "COPY failed" durante docker build

**Causa:** Archivo especificado en Dockerfile no existe

**Solución:**
1. Verifica que todos los archivos existen:
   ```bash
   ls -la heart_disease_model.joblib
   ls -la heart_model_wrapper.py
   ls -la api_heart.py
   ls -la requirements.txt
   ```
2. Verifica que estás ejecutando `docker build` desde el directorio correcto (Tema_07)

### Error: "no space left on device" durante docker build

**Causa:** Disco lleno o Docker ha consumido mucho espacio

**Solución:**
```bash
# Limpiar imágenes no usadas
docker system prune -a

# Ver uso de espacio
docker system df
```

### Error: Contenedor se detiene inmediatamente después de iniciarlo

**Causa:** Error en la aplicación al iniciar

**Solución:**
```bash
# Ver logs del contenedor
docker logs heart-api-container

# Ver logs en tiempo real
docker logs -f heart-api-container

# Ejecutar contenedor sin -d para ver errores
docker run -p 8000:8000 heart-api:v1.0
```

## 🔧 Problemas con Requests desde Python

### Error: "ConnectionError: Failed to establish a new connection"

**Causa:** La API no está corriendo o la URL es incorrecta

**Solución:**
1. Verifica que uvicorn o el contenedor está corriendo
2. Verifica la URL: `http://localhost:8000`
3. Prueba con curl primero:
   ```bash
   curl http://localhost:8000/health
   ```

### Error: "ReadTimeout: HTTPConnectionPool read timed out"

**Causa:** La API tarda demasiado en responder

**Solución:**
- Incrementa el timeout:
```python
response = requests.get(url, timeout=30)  # 30 segundos
```

## 📦 Problemas Generales

### El modelo tiene muy bajo rendimiento (accuracy < 70%)

**Causa Normal:** Dataset sintético o pequeño

**No es un problema:** El objetivo del ejercicio es el proceso de despliegue, no el rendimiento del modelo. En un proyecto real, dedicarías más tiempo a mejorar el modelo.

### Los notebooks no cargan o se congela Jupyter

**Solución:**
```bash
# Reiniciar kernel de Jupyter
# En Jupyter: Kernel → Restart

# O reiniciar Jupyter completamente
jupyter notebook stop
jupyter notebook
```

### Error al importar módulos propios (heart_model_wrapper)

**Causa:** Python no encuentra el módulo

**Solución:**
1. Verifica que el archivo existe en el mismo directorio
2. Verifica que estás ejecutando desde el directorio correcto
3. Si es necesario, añade el directorio al PATH:
```python
import sys
sys.path.append('.')
```

## 🆘 Estrategia General de Debugging

**Cuando algo falle:**

1. **Lee el mensaje de error completo**
   - Los errores suelen decir exactamente qué está mal

2. **Verifica los logs**
   - uvicorn: directamente en la terminal
   - Docker: `docker logs <container>`

3. **Prueba paso a paso**
   - No intentes probar todo junto
   - Verifica cada componente por separado

4. **Usa print() o logging**
   - Añade prints para ver qué está pasando
   ```python
   print(f"Tipo de datos: {type(patient_data)}")
   print(f"Contenido: {patient_data}")
   ```

5. **Revisa la documentación**
   - FastAPI docs: http://localhost:8000/docs
   - Swagger UI muestra exactamente qué espera cada endpoint

6. **Google/Stack Overflow**
   - Copia el mensaje de error exacto
   - Busca: "fastapi [tu error]"
   - Busca: "docker [tu error]"

## 💬 ¿Aún atascado?

**Durante la clase:**
- Levanta la mano y pregunta al instructor
- Trabaja con tus compañeros

**Después de la clase:**
- Revisa el cuaderno teórico: `Empaquetado_y_Despliegue_de_Modelos_ML.ipynb`
- Consulta la documentación oficial
- Busca en Stack Overflow
- Pregunta en comunidades de ML/Python

**Recuerda:** Los errores son parte del proceso de aprendizaje. Cada error que resuelves te hace mejor desarrollador de ML. 🚀